**3.3.2 Import Library dan Dataset**

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.preprocessing import RobustScaler
from yellowbrick.cluster import KElbowVisualizer

In [ ]:
# Load dataset
#link = 'https://drive.google.com/file/d/1PuUpymCAaC3FIO1spmMFHp3fz7nWVv7k/view?usp=sharing'
#df = pd.read_csv(f'https://drive.google.com/uc?export=download&id={link.split('/')[-2]}')
df = pd.read_excel('olah_data_fix.xlsx')

**3.3.3 Memahami Struktur Dataset**

In [ ]:
df.info()

In [ ]:
df.describe()

**3.3.4 Pembersihan Data**


In [ ]:
# Mengecek nilai kosong
df.isnull().sum()

In [ ]:
# Mengecek duplikat
df.duplicated(subset=['Provinsi', 'Tahun', 'Bulan']).sum()

**3.3.5 Eksplorasi Variabel Utama**

In [ ]:
numeric_df =['outflow_tunai', 'kartu_atm_debet', 'Server_Based', 'SKNBI_Asal']

plt.figure(figsize=(10,6))
for i, column in enumerate(numeric_df, 1):
    plt.subplot(2, 2, i)
    sns.histplot(df[column], bins=30, kde=True, color='blue')
    plt.title(f'Distribusi {column}')
plt.tight_layout()
plt.show()

In [ ]:
# Analisis hubungan antar variabel numerik
plt.figure(figsize=(8,6))
sns.heatmap(df[numeric_df].corr(), annot=True, cmap='coolwarm')
plt.title("Korelasi Antar Variabel Numerik", fontsize=16)
plt.show()

In [ ]:
# menghitung outlier tiap kolom numerik
for column in numeric_df:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    print(f"Outliers pada kolom {column}: {outliers.shape[0]}")

In [ ]:
# Boxplot tiap kolom numerik
numeric_df = ['outflow_tunai', 'kartu_atm_debet', 'Server_Based', 'SKNBI_Asal']
for column in numeric_df:
    plt.figure(figsize=(8, 6))
    sns.boxplot(x=df[column])
    plt.title(f'Boxplot {column}')
    plt.show()

**3.3.6 Transformasi Logaritmik**

In [ ]:
df_agg = df.groupby(['Provinsi', 'Tahun'])[numeric_df].mean().reset_index()
print(df_agg.shape)
print(df_agg.head())

In [ ]:
skew_before = df_agg[numeric_df].skew()
for col in numeric_df:
    df_agg[col] = np.log1p(df_agg[col])
skew_after = df_agg[numeric_df].skew()
skewness_df = pd.DataFrame({
    'Sebelum Log-Transform': skew_before.round(2),
    'Setelah Log-Transform': skew_after.round(2)
})

skewness_df

In [ ]:
df_agg_before_log = df.groupby(['Provinsi', 'Tahun'])[numeric_df].mean().reset_index()

fig, axes = plt.subplots(len(numeric_df), 2, figsize=(15, 4*len(numeric_df)))
for i, col in enumerate(numeric_df):
    sns.histplot(df_agg_before_log[col], bins=30, kde=True, color='#D85A30', ax=axes[i, 0])
    axes[i, 0].set_title(f'{col} (Sebelum Log-Transform)')
    axes[i, 0].set_xlabel(col)
    axes[i, 0].set_ylabel('Frekuensi')
    sns.histplot(df_agg[col], bins=30, kde=True, color='#1D9E75', ax=axes[i, 1])
    axes[i, 1].set_title(f'{col} (Setelah Log-Transform)')
    axes[i, 1].set_xlabel(col)
    axes[i, 1].set_ylabel('Frekuensi')

plt.tight_layout()
plt.show()

**3.3.7 Standarisasi Data**

In [ ]:
# Scaling dengan Robust scaler
scaler = RobustScaler()
X_all = scaler.fit_transform(df_agg[numeric_df])
df_agg['scaled_idx'] = range(len(df_agg))

**3.4 Membangun Model Clustering**

In [ ]:
# Elbow Method
model = KMeans(random_state=42, n_init=10)
visualizer = KElbowVisualizer(model, k=(2, 8))
visualizer.fit(X_all)
visualizer.show()
optimal_k = visualizer.elbow_value_
print(f"Jumlah klaster optimal: {optimal_k}")

In [ ]:
years = sorted(df_agg['Tahun'].unique())
results_list = []
year_ref = years[0]
df_ref = df_agg[df_agg['Tahun'] == year_ref].copy()
idx_ref = df_ref['scaled_idx'].values
X_ref = X_all[idx_ref]

In [ ]:
# Kmeans
kmeans_ref = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
labels_ref = kmeans_ref.fit_predict(X_ref)
df_ref['Cluster_raw'] = labels_ref
results_list.append(df_ref)

In [ ]:
# Centroid cluster
centroids_df = pd.DataFrame(kmeans_ref.cluster_centers_, columns=numeric_df)
display(centroids_df.round(2))

In [ ]:
# Clustering tahunan dengan relabeling cluster
for year in years[1:]:
    df_yr = df_agg[df_agg['Tahun'] == year].copy()
    idx_yr = df_yr['scaled_idx'].values
    X_yr = X_all[idx_yr]
    km_yr = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    labels_yr = km_yr.fit_predict(X_yr)
    cost_matrix = euclidean_distances(km_yr.cluster_centers_, kmeans_ref.cluster_centers_)
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    label_map = {row: col for row, col in zip(row_ind, col_ind)}
    df_yr['Cluster_raw'] = [label_map[label] for label in labels_yr]
    results_list.append(df_yr)

df_clustered_yearly = pd.concat(results_list, ignore_index=True)

**3.5 Evaluasi Model Clustering**

**3.5.1 Validasi Multi-Metrik**

In [ ]:
# Evaluasi cluster antar waktu
metrics_results = []

for year in years:
    mask = df_clustered_yearly['Tahun'] == year
    idx = df_clustered_yearly.loc[mask, 'scaled_idx'].values
    X_yr = X_all[idx]
    labels_yr = df_clustered_yearly.loc[mask, 'Cluster_raw'].values
    if len(np.unique(labels_yr)) > 1 and len(X_yr) > 1:
        centroids = {
            cluster: X_yr[labels_yr == cluster].mean(axis=0)
            for cluster in np.unique(labels_yr)
        }
        wcss_score = sum(
            np.sum((X_yr[i] - centroids[labels_yr[i]])**2)
            for i in range(len(X_yr))
        )
        sil_score = silhouette_score(X_yr, labels_yr)
        ch_score = calinski_harabasz_score(X_yr, labels_yr)
        db_score = davies_bouldin_score(X_yr, labels_yr)
    else:
        wcss_score = sil_score = ch_score = db_score = np.nan
    metrics_results.append({
        'Tahun': year,
        'WCSS': wcss_score,
        'Silhouette Score': sil_score,
        'Calinski-Harabasz Score': ch_score,
        'Davies-Bouldin Score': db_score
    })
df_metrics = pd.DataFrame(metrics_results).set_index('Tahun')
display(df_metrics.round(4))

In [ ]:
# Buat visualisasi dari tabel Evaluasi Stabilitas Klaster per Tahun
plt.figure(figsize=(15, 10))
metrics_to_plot = df_metrics.columns

for i, metric in enumerate(metrics_to_plot, 1):
    plt.subplot(len(metrics_to_plot), 1, i)
    sns.lineplot(x=df_metrics.index, y=metric, data=df_metrics, marker='o')
    plt.title(f'{metric} per Tahun', fontsize=12)
    plt.xlabel('Tahun', fontsize=10)
    plt.ylabel(metric, fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.7)
    for x, y in zip(df_metrics.index, df_metrics[metric]):
        plt.text(x, y, f'{y:.2f}', ha='right', va='bottom', fontsize=8)

plt.suptitle('Evaluasi Stabilitas Klaster k=4 per Tahun', fontsize=16, y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

**3.5.2 Interpretasi Hasil Clustering**

In [ ]:
# Perbandingan hasil cluster provinsi per tahun
comparison_table_raw = df_clustered_yearly.pivot_table(
    index='Provinsi',
    columns='Tahun',
    values='Cluster_raw',
    aggfunc='first'
)
display(comparison_table_raw)

In [ ]:
# Label semantik
label_mapping = {
    3: 'Digital Rendah',
    0: 'Digital Menengah',
    1: 'Digital Maju',
    2: 'Digital Spesialis Non-Tunai'
}
df_clustered_yearly['Target_Semantic'] = df_clustered_yearly['Cluster_raw'].map(label_mapping)

In [ ]:
# Perbandingan hasil cluster provinsi per tahun dengan label semantik
comparison_table = df_clustered_yearly.pivot_table(
    index='Provinsi',
    columns='Tahun',
    values='Target_Semantic',
    aggfunc='first'
)
display(comparison_table)

**3.5.3 Analisis Perubahan Cluster Antar Tahun**

In [ ]:
change_records = []

for i in range(len(years) - 1):
    y1, y2 = years[i], years[i + 1]
    changed = comparison_table[comparison_table[y1] != comparison_table[y2]]

    for prov, row in changed.iterrows():
        change_records.append({
            'Provinsi': prov,
            'Tahun Awal': y1,
            'Cluster Awal': row[y1],
            'Tahun Akhir': y2,
            'Cluster Akhir': row[y2]
        })

if change_records:
    df_changes = pd.DataFrame(change_records)
    display(df_changes)
else:
    print("Tidak ada perubahan klaster antar tahun.")

**3.6 Deployment**

**3.6.1 Visualisasi PCA**

In [ ]:
# Dimensi PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_all)
df_clustered_yearly['PCA1'] = pca_result[df_clustered_yearly['scaled_idx'], 0]
df_clustered_yearly['PCA2'] = pca_result[df_clustered_yearly['scaled_idx'], 1]

In [ ]:
# Visualisasi PCA
ordered_labels = [
    'Digital Rendah',
    'Digital Menengah',
    'Digital Maju',
    'Digital Spesialis Non-Tunai'
]

df_clustered_yearly['Target_Semantic'] = pd.Categorical(
    df_clustered_yearly['Target_Semantic'],
    categories=ordered_labels,
    ordered=True
)

semantic_color_map = {
    'Digital Rendah': '#e41a1c',
    'Digital Menengah': '#377eb8',
    'Digital Maju': '#4daf4a',
    'Digital Spesialis Non-Tunai': '#984ea3'
}

plt.figure(figsize=(10, 8))
sns.scatterplot(data=df_clustered_yearly, x='PCA1', y='PCA2', hue='Target_Semantic',
                palette=semantic_color_map,s=100, alpha=0.7, edgecolor='white')

plt.title('Visualisasi Klaster Menggunakan PCA')
plt.xlabel(f'Komponen Utama 1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Komponen Utama 2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.legend(title='Klaster Semantik', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# **3.6.2 Visualisasi Distribusi Cluster per Tahun**

In [ ]:
cluster_counts = (
    df_clustered_yearly.groupby('Cluster_raw')
    .size()
    .reset_index(name='Count')
)

cluster_counts['Cluster_raw'] = cluster_counts['Cluster_raw'].map(label_mapping)

ordered_labels = [
    'Digital Rendah',
    'Digital Menengah',
    'Digital Maju',
    'Digital Spesialis Non-Tunai'
]

cluster_counts = (
    cluster_counts
    .set_index('Cluster_raw')
    .loc[ordered_labels]
)

plt.figure(figsize=(8, 6))

ax = sns.barplot(
    data=cluster_counts,
    x=cluster_counts.index,
    y='Count',
    hue=cluster_counts.index,
    palette=[semantic_color_map[label] for label in cluster_counts.index],
    legend=False
)

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f')

plt.title('Distribusi Klaster Keseluruhan')
plt.xlabel('Klaster Semantik')
plt.ylabel('Jumlah Provinsi')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
cluster_counts = (
    df_clustered_yearly
    .groupby(['Tahun', 'Target_Semantic'])
    .size()
    .reset_index(name='Count')
)

cluster_counts['Target_Semantic'] = pd.Categorical(
    cluster_counts['Target_Semantic'],
    categories=ordered_labels,
    ordered=True
)

cluster_counts = cluster_counts.sort_values(['Tahun', 'Target_Semantic'])

plt.figure(figsize=(12, 7))

ax = sns.barplot(
    data=cluster_counts,
    x='Tahun',
    y='Count',
    hue='Target_Semantic',
    palette=semantic_color_map,
    order=sorted(cluster_counts['Tahun'].unique())
)

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f')

plt.title('Distribusi Klaster per Tahun')
plt.xlabel('Tahun')
plt.ylabel('Jumlah Provinsi')
plt.legend(title='Klaster Semantik', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Buat visualisasi menarik lagi dari hasil cluster

# Recreate centroids_df using kmeans_ref.cluster_centers_ and numeric_df
centroids = kmeans_ref.cluster_centers_
centroids_df = pd.DataFrame(centroids, columns=numeric_df)
centroids_df['Cluster'] = centroids_df.index

# Create a heatmap of the cluster centroids to visualize the characteristics of each semantic cluster
plt.figure(figsize=(10, 6))
sns.heatmap(
    centroids_df.set_index('Cluster').rename(index=label_mapping),
    annot=True,
    cmap='viridis',
    fmt=".2f"
)
plt.title('Heatmap Karakteristik Centroid Klaster Semantik')
plt.xlabel('Fitur Skala')
plt.ylabel('Klaster Semantik')
plt.yticks(rotation=0)
plt.show()

In [ ]:
#simpan final clustering
df_clustered_yearly.to_csv('hasil_clustering_final.csv', index=False)
print('hasil_clustering_final.csv tersimpan —', df_clustered_yearly.shape)

End of Code.

In [ ]:
# pie chart proporsi cluster
cluster_counts = df_clustered_yearly['Target_Semantic'].value_counts()
plt.figure(figsize=(8, 8))
plt.pie(cluster_counts, labels=cluster_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Proporsi Cluster')
plt.show()

## Kesimpulan Clustering

Analisis klaster telah berhasil mengelompokkan provinsi-provinsi berdasarkan perilaku transaksi digital mereka ke dalam empat kategori semantik:

1.  **Digital Rendah**: Klaster ini didominasi oleh provinsi-provinsi dengan tingkat outflow tunai, kartu ATM/debet, Server-Based, dan SKNBI Asal yang rendah. Mereka cenderung memiliki adopsi digital yang terbatas dalam transaksi keuangan.
2.  **Digital Menengah**: Provinsi dalam klaster ini menunjukkan tingkat aktivitas transaksi digital yang moderat di semua kategori. Mereka berada di tengah-tengah spektrum adopsi digital.
3.  **Digital Maju**: Klaster ini mencakup provinsi-provinsi dengan adopsi digital yang tinggi di semua kategori transaksi. Mereka aktif dalam penggunaan instrumen digital untuk berbagai keperluan finansial.
4.  **Digital Spesialis Non-Tunai**: Klaster unik ini menunjukkan outflow tunai yang sangat rendah, tetapi sangat tinggi dalam penggunaan kartu ATM/debet, Server-Based, dan SKNBI Asal. Ini mengindikasikan provinsi yang sangat mengandalkan transaksi non-tunai dan mungkin memiliki inisiatif khusus untuk mengurangi penggunaan uang tunai.

**Proporsi Klaster:**
*   Mayoritas provinsi (55.9%) termasuk dalam kategori **Digital Menengah**, menunjukkan bahwa sebagian besar wilayah memiliki tingkat adopsi digital yang stabil namun belum mencapai tingkat optimal.
*   **Digital Rendah** mencakup 29.4% provinsi, menandakan adanya wilayah yang masih memerlukan dorongan signifikan dalam adopsi digital.
*   **Digital Maju** dan **Digital Spesialis Non-Tunai** masing-masing hanya 11.8% dan 2.9%, menunjukkan bahwa provinsi yang benar-benar memimpin atau memiliki spesialisasi digital masih merupakan minoritas.

**Perubahan dan Stabilitas Antar Tahun (2021-2025):**

Evaluasi stabilitas klaster menunjukkan bahwa sebagian besar provinsi mempertahankan kategori klaster mereka dari tahun ke tahun. Namun, terdapat beberapa pergerakan yang menarik:

*   **Pergerakan Positif:** Beberapa provinsi menunjukkan peningkatan dari 'Digital Rendah' ke 'Digital Menengah' atau dari 'Digital Menengah' ke 'Digital Maju', menandakan keberhasilan upaya peningkatan literasi dan infrastruktur digital.
*   **Pergerakan Negatif:** Ada juga beberapa kasus di mana provinsi mengalami penurunan kategori, misalnya dari 'Digital Menengah' menjadi 'Digital Rendah', yang mungkin disebabkan oleh berbagai faktor seperti perubahan ekonomi regional atau kurangnya inovasi digital.

Secara keseluruhan, temuan ini memberikan wawasan berharga bagi pembuat kebijakan untuk merancang strategi yang ditargetkan dalam mendorong inklusi keuangan dan adopsi pembayaran digital di seluruh Indonesia, dengan memperhatikan karakteristik unik setiap klaster.